# 01 · Dataset understanding
Checks the raw file before any modelling: structure, target balance, data quality, unusual values and duplicate columns. Nothing here changes the data; every decision it leads to is applied in notebook 02.

## 1. First inspection: structure, target and data quality
**What it does:** loads `data.csv`, strips the leading spaces from column names, then reports the table size, data types, how many records are bankrupt, missing values, duplicate rows, columns with only one or two distinct values, and columns with values outside 0–1.

**Why:** confirms the file is complete and shows the three issues that shape the project: a 3.2% bankrupt class, a constant column, and 24 columns with values above 1.

In [ ]:
import pandas as pd

df = pd.read_csv("data/data.csv")
df.columns = df.columns.str.strip()       # remove the leading spaces in names

print("Shape:", df.shape)
print("\nData types:\n", df.dtypes.value_counts())
print("\nTarget counts:\n", df["Bankrupt?"].value_counts())
print("\nTarget %:\n", (df["Bankrupt?"].value_counts(normalize=True) * 100).round(2))
print("\nMissing values (total):", df.isna().sum().sum())
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate rows ignoring target:", df.drop(columns="Bankrupt?").duplicated().sum())

nunique = df.nunique()
print("\nColumns with only 1 value:", nunique[nunique == 1].index.tolist())
print("Columns with only 2 values:", nunique[nunique == 2].index.tolist())

out_of_range = ((df < 0) | (df > 1)).sum()
print("\nColumns with values outside [0, 1]:", (out_of_range > 0).sum())
print(out_of_range[out_of_range > 0].sort_values(ascending=False))

Shape: (6819, 96)

Data types:
 float64    93
int64       3
Name: count, dtype: int64

Target counts:
 Bankrupt?
0    6599
1     220
Name: count, dtype: int64

Target %:
 Bankrupt?
0   96.77
1    3.23
Name: proportion, dtype: float64

Missing values (total): 0
Exact duplicate rows: 0
Duplicate rows ignoring target: 0

Columns with only 1 value: ['Net Income Flag']
Columns with only 2 values: ['Bankrupt?', 'Liability-Assets Flag']

Columns with values outside [0, 1]: 24
Total Asset Growth Rate                  6017
Cash Turnover Rate                       4239
Research and development expense rate    4033
Inventory Turnover Rate (times)          2946
Quick Asset Turnover Rate                2383
Operating Expense Rate                   2282
Current Asset Turnover Rate              1234
Fixed Assets Turnover Frequency          1228
Interest-bearing debt interest rate       221
Long-term Liability to Current Assets     109
Inventory/Current Liability                99
Cash/Current Liabili

## 2. Liability-Assets Flag and the size of the out-of-range values
**What it does:** (1) cross-tabulates `Liability-Assets Flag` against bankruptcy to show the bankruptcy rate when liabilities exceed assets; (2) for each of the 24 columns with values above 1, counts those values and shows their median, 99th percentile, maximum, and how many belong to bankrupt records.

**Why:** tells us whether the large values are a different scale for the whole column or a few isolated extremes, and whether they are linked to bankruptcy before deciding how to treat them.

In [ ]:
# 1. Liability-Assets Flag vs bankruptcy
print("Liability-Assets Flag vs Bankrupt? (counts):")
print(pd.crosstab(df["Liability-Assets Flag"], df["Bankrupt?"], margins=True))
print("\nBankruptcy rate within each flag value (%):")
print((pd.crosstab(df["Liability-Assets Flag"], df["Bankrupt?"], normalize="index") * 100).round(1))

# 2. Size of the out-of-range values
cols = out_of_range[out_of_range > 0].index
summary = pd.DataFrame({
    "n_above_1":        (df[cols] > 1).sum(),
    "n_negative":       (df[cols] < 0).sum(),
    "median":           df[cols].median(),
    "99th_percentile":  df[cols].quantile(0.99),
    "max":              df[cols].max(),
    "bankrupt_among_above_1": [df.loc[df[c] > 1, "Bankrupt?"].sum() for c in cols],
})
pd.set_option("display.float_format", "{:,.4g}".format)
summary.sort_values("n_above_1", ascending=False)

Liability-Assets Flag vs Bankrupt? (counts):
Bankrupt?                 0    1   All
Liability-Assets Flag                 
0                      6597  214  6811
1                         2    6     8
All                    6599  220  6819

Bankruptcy rate within each flag value (%):
Bankrupt?                0   1
Liability-Assets Flag         
0                     96.9 3.1
1                       25  75


,n_above_1,n_negative,median,99th_percentile,max,bankrupt_among_above_1
Total Asset Growth Rate,6017,0,6.4e+09,9.758e+09,9.99e+09,212
Cash Turnover Rate,4239,0,1.08e+09,9.62e+09,1e+10,184
Research and development expense rate,4033,0,5.09e+08,9.44e+09,9.98e+09,120
Inventory Turnover Rate (times),2946,0,0.0007647,9.73e+09,9.99e+09,118
Quick Asset Turnover Rate,2383,0,0.0002248,9.83e+09,1e+10,97
Operating Expense Rate,2282,0,0.0002778,9.79e+09,9.99e+09,67
Current Asset Turnover Rate,1234,0,0.0001988,9.788e+09,1e+10,53
Fixed Assets Turnover Frequency,1228,0,0.0005931,9.46e+09,9.99e+09,84
Interest-bearing debt interest rate,221,0,0.000321,7.582e+08,9.9e+08,2
Long-term Liability to Current Assets,109,0,0.001975,1.398e+09,9.54e+09,2


## 3. Inspecting the records that define the column bounds
**What it does:** for records 2490, 6610 and 5015 (row positions in the file, not company IDs), shows their target, flag, how many of their values are exactly 0, exactly 1 or above 1, which columns hold large values, and a full table of every ratio with its percentile rank compared with the column median.

**Why:** these records sit at the minimum or maximum of many columns. Percentile ranks show whether their extremes form a coherent financial profile or look like data errors, before deciding to keep or remove them.

In [ ]:
import numpy as np
pd.set_option("display.max_rows", 120)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:.4g}".format)

TARGET = "Bankrupt?"
features = [c for c in df.columns if c != TARGET]
rows = [2490, 6610, 5015]

# Percentile rank of every value inside its own column (0 = lowest company, 100 = highest)
pct = df[features].rank(pct=True) * 100

summary = pd.DataFrame({
    "Bankrupt?":             df.loc[rows, TARGET],
    "Liability-Assets Flag": df.loc[rows, "Liability-Assets Flag"],
    "n_exactly_0":           (df.loc[rows, features] == 0).sum(axis=1),
    "n_exactly_1":           (df.loc[rows, features] == 1).sum(axis=1),
    "n_greater_than_1":      (df.loc[rows, features] > 1).sum(axis=1),
})
print(summary)

for r in rows:
    big = df.loc[r, features][df.loc[r, features] > 1]
    print(f"\nRow {r} - columns with values > 1:")
    print(big.map("{:.3g}".format).to_string() if len(big) else "   none")

full = pd.DataFrame({"median_all": df[features].median()})
for r in rows:
    full[f"row_{r}"] = df.loc[r, features]
    full[f"pct_{r}"] = pct.loc[r].round(1)
full

      Bankrupt?  Liability-Assets Flag  n_exactly_0  n_exactly_1  n_greater_than_1
2490          0                      0           15            4                 4
6610          0                      0            4            9                 1
5015          0                      0            9            9                 1

Row 2490 - columns with values > 1:
Total Asset Growth Rate            8.58e+09
Total debt/Total net worth         4.74e+08
Inventory Turnover Rate (times)    1.91e+07
Cash Turnover Rate                 6.88e+09

Row 6610 - columns with values > 1:
Inventory/Current Liability    6.79e+09

Row 5015 - columns with values > 1:
Inventory Turnover Rate (times)    3.82e+07


,median_all,row_2490,pct_2490,row_6610,pct_6610,row_5015,pct_5015
ROA(C) before interest and depreciation before interest,0.5027,0.4122,4.9,1,100,0.4469,11.2
ROA(A) before interest and % after tax,0.5598,0.4743,7,0.9471,100,0.5154,16.3
ROA(B) before interest and depreciation after tax,0.5523,0.4577,4.8,0.9578,100,0.4982,11.8
Operating Gross Margin,0.606,0,0,0.6407,97.6,0.6098,63.9
Realized Sales Gross Margin,0.606,0,0,0.6407,97.6,0.6098,64.1
Operating Profit Rate,0.999,0.988,0.2,0.9995,99.6,0,0
Pre-tax net Interest Rate,0.7975,0.7843,0.2,0.798,98.4,0,0
After-tax net Interest Rate,0.8094,0.7966,0.2,0.8098,97.5,0.08429,0
Non-industry income and expenditure/revenue,0.3035,0.3035,50.2,0.3035,44.3,1,100
Continuous interest rate (after tax),0.7816,0.7679,0.2,0.7821,97.7,0,0


## 4. Internal consistency check on related ratios
**What it does:** compares the percentile ranks of the same three records on pairs of ratios that should move together ("same") or in opposite directions ("opposite"), for example debt ratio vs net worth to assets.

**Why:** if a record is extreme but its related ratios agree, it is a genuine extreme company; large contradictions would point to errors.

In [ ]:
pairs = [
    ("Debt ratio %", "Net worth/Assets", "opposite"),
    ("Liability to Equity", "Equity to Liability", "opposite"),
    ("ROA(A) before interest and % after tax", "Net Income to Total Assets", "same"),
    ("ROA(C) before interest and depreciation before interest",
     "ROA(B) before interest and depreciation after tax", "same"),
    ("Current Liability to Assets", "Debt ratio %", "same"),
    ("Current Assets/Total Assets", "Quick Assets/Total Assets", "same"),
    ("Current Ratio", "Current Liability to Current Assets", "opposite"),
    ("Persistent EPS in the Last Four Seasons", "Per Share Net profit before tax (Yuan ¥)", "same"),
    ("Cash Flow to Total Assets", "CFO to Assets", "same"),
    ("Total Asset Turnover", "Net Worth Turnover Rate (times)", "same"),
]
check = []
for a, b, rel in pairs:
    for r in rows:
        check.append({"row": r, "feature_A": a[:35], "feature_B": b[:35], "expected": rel,
                      "pct_A": round(pct.loc[r, a], 1), "pct_B": round(pct.loc[r, b], 1)})
pd.DataFrame(check).sort_values(["row", "expected"])

,row,feature_A,feature_B,expected,pct_A,pct_B
0,2490,Debt ratio %,Net worth/Assets,opposite,0.1,99.9
3,2490,Liability to Equity,Equity to Liability,opposite,0.2,99.9
18,2490,Current Ratio,Current Liability to Current Assets,opposite,100,0
6,2490,ROA(A) before interest and % after,Net Income to Total Assets,same,7,8.8
9,2490,ROA(C) before interest and deprecia,ROA(B) before interest and deprecia,same,4.9,4.8
12,2490,Current Liability to Assets,Debt ratio %,same,0,0.1
15,2490,Current Assets/Total Assets,Quick Assets/Total Assets,same,10.5,26.9
21,2490,Persistent EPS in the Last Four Sea,Per Share Net profit before tax (Yu,same,8.3,8.9
24,2490,Cash Flow to Total Assets,CFO to Assets,same,13.1,14.1
27,2490,Total Asset Turnover,Net Worth Turnover Rate (times),same,0.2,0.2


## 5. Checking for duplicate features
**What it does:** tests whether `Current Liabilities/Liability` and `Current Liability to Liability` are identical row by row (equality, largest difference, correlations, distributions), then scans every pair of the 95 features for exact copies.

**Why:** duplicate columns add no information and split feature importance between twins, so they are identified from the data, not from their names.

In [ ]:
a, b = "Current Liabilities/Liability", "Current Liability to Liability"
print("Identical for every row:     ", (df[a] == df[b]).all())
print("Rows where they differ:      ", (df[a] != df[b]).sum())
print("Largest absolute difference: ", (df[a] - df[b]).abs().max())
print("Pearson correlation:         ", df[a].corr(df[b]))
print("Spearman (rank) correlation: ", df[a].corr(df[b], method="spearman"))
print(pd.DataFrame({a: df[a].describe(), b: df[b].describe()}))

# Scan every pair of features for other exact duplicates
dups = [(x, y) for i, x in enumerate(features) for y in features[i+1:] if df[x].equals(df[y])]
print("\nAll exactly identical feature pairs:")
for x, y in dups:
    print("  ", x, "==", y)

Identical for every row:      True
Rows where they differ:       0
Largest absolute difference:  0.0
Pearson correlation:          1.0
Spearman (rank) correlation:  0.9999999999999999
       Current Liabilities/Liability  Current Liability to Liability
count                           6819                            6819
mean                          0.7616                          0.7616
std                           0.2067                          0.2067
min                                0                               0
25%                            0.627                           0.627
50%                           0.8069                          0.8069
75%                            0.942                           0.942
max                                1                               1

All exactly identical feature pairs:
   Current Liabilities/Liability == Current Liability to Liability
   Current Liabilities/Equity == Current Liability to Equity


## 6. Which records define the 0 and 1 bounds?
**What it does:** for every continuous column, finds the single record holding the exact 0 or 1, counts how many bounds each record holds, compares the bankruptcy rate of those records with the overall 3.2%, and simulates how concentrated the bounds would be if they were assigned at random (seed 42).

**Why:** shows whether a few records set the scale of many columns. The concentration turned out to come from ratios sharing the same building blocks (revenue, liabilities, equity), not from errors.

In [ ]:
cont = [c for c in features if df[c].nunique() > 2]           # skip the two flag columns
holders = []
for c in cont:
    for bound in (0, 1):
        who = df.index[df[c] == bound]
        if len(who) == 1:                                      # only bounds held by ONE row
            holders.append({"column": c, "bound": bound, "row": who[0]})
holders = pd.DataFrame(holders)
print("Bounds held by a single row:", len(holders), "| distinct rows holding them:", holders["row"].nunique())

per_row = holders.groupby("row").agg(n_bounds=("column", "size"),
                                      n_at_0=("bound", lambda s: (s == 0).sum()),
                                      n_at_1=("bound", lambda s: (s == 1).sum()))
per_row["Bankrupt?"] = df.loc[per_row.index, TARGET]
per_row = per_row.sort_values("n_bounds", ascending=False)
print(per_row.head(15))
print("\nHow many rows hold k bounds:")
print(per_row["n_bounds"].value_counts().sort_index())
print("\nBankruptcy rate among rows holding >= 3 bounds:",
      round(per_row.loc[per_row["n_bounds"] >= 3, "Bankrupt?"].mean() * 100, 1), "%  (overall 3.2%)")

# Benchmark: what if each bound were held by a randomly chosen row?
rng = np.random.default_rng(42)
sims = [np.bincount(rng.integers(0, len(df), len(holders))).max() for _ in range(1000)]
print("Random benchmark - max bounds held by one row: typical", int(np.median(sims)),
      "| highest in 1,000 simulations", max(sims))

Bounds held by a single row: 171 | distinct rows holding them: 73
      n_bounds  n_at_0  n_at_1  Bankrupt?
row                                      
5015        11       3       8          0
2490        10       7       3          0
6613        10       8       2          0
2440         9       4       5          1
1684         7       6       1          0
4447         7       4       3          0
6610         6       0       6          0
2393         6       2       4          0
1869         5       5       0          1
1870         4       3       1          1
2411         4       1       3          0
2605         4       4       0          0
6391         4       2       2          0
4637         3       0       3          0
2363         3       0       3          0

How many rows hold k bounds:
n_bounds
1     41
2     14
3      5
4      4
5      1
6      2
7      2
9      1
10     2
11     1
Name: count, dtype: int64

Bankruptcy rate among rows holding >= 3 bounds: 16.7 %  (overall